In [0]:
#====================================================================================
# VIDA CRM Lakehouse
# Silver Layer - Data Transformation
# Author: Emmanuel Quesada Gómez
#====================================================================================

from pyspark.sql.functions import col, trim, upper, lower, when, to_date, current_timestamp

BRONZE_PATH = "workspace.default"
SILVER_PATH = "workspace.default"

print("Libraries loaded successfully")



Libraries loaded successfully


In [0]:
#====================================================================================
# Silver - dim_volunteer
#====================================================================================

df_volunteer = spark.table(f"{BRONZE_PATH}.bronze_volunteer")

df_silver_volunteer = df_volunteer \
  .withColumn("first_name", trim(col("first_name"))) \
  .withColumn("last_name", trim(col("last_name"))) \
  .withColumn("email", lower(trim(col("email")))) \
  .withColumn("gender", trim(col("gender"))) \
  .withColumn("nationality", trim(col("nationality"))) \
  .withColumn("birth_date", to_date(col("birth_date"), "yyyy-MM-dd")) \
  .withColumn("start_date", to_date(col("start_date"), "yyyy-MM-dd")) \
  .withColumn("status", trim(col("status"))) \
  .withColumn("education_level", trim(col("education_level"))) \
  .withColumn("ingestion_timestamp", current_timestamp()) \
  .dropDuplicates(["volunteer_id"]) \
  .filter(col("volunteer_id").isNotNull())
  
df_silver_volunteer.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_PATH}.silver_volunteer")
print(f"silver_volunteer: {df_silver_volunteer.count()} records")

silver_volunteer: 8000 records


In [0]:
#====================================================================================
# Silver - dim_geography
#====================================================================================

df_geography = spark.table(f"{BRONZE_PATH}.bronze_geography")

df_silver_geography = df_geography \
    .withColumn("country", trim(col("country"))) \
    .withColumn("region", trim(col("region"))) \
    .withColumn("capital_city", trim(col("capital_city"))) \
    .withColumn("city", trim(col("city"))) \
    .withColumn("rural_urban", trim(col("rural_urban"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .dropDuplicates(["geography_id"]) \
    .filter(col("geography_id").isNotNull())

df_silver_geography.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_PATH}.silver_geography")
print(f"silver_geography: {df_silver_geography.count()} records")

silver_geography: 250 records


In [0]:
#====================================================================================
# Silver - dim_organization
#====================================================================================

df_organization = spark.table(f"{BRONZE_PATH}.bronze_organization")

df_silver_organization = df_organization \
    .withColumn("organizarion_name", trim(col("organization_name"))) \
    .withColumn("org_type", trim(col("org_type"))) \
    .withColumn("contact_email", lower(trim(col("contact_email")))) \
    .withColumn("active", col("active").cast("boolean")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .dropDuplicates(["organization_id"]) \
    .filter(col("organization_id").isNotNull())

df_silver_organization.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_PATH}.silver_organization")
print(f"silver_organiztion: {df_silver_organization.count()} records")

silver_organiztion: 800 records


In [0]:
#====================================================================================
# Silver - dim_project
#====================================================================================

df_project = spark.table(f"{BRONZE_PATH}.bronze_project")

df_silver_project = df_project \
    .withColumn("project_name", trim(col("project_name"))) \
    .withColumn("project_type", trim(col("project_type"))) \
    .withColumn("start_date", to_date(col("start_date"), "yyyy-MM-dd")) \
    .withColumn("end_date", to_date(col("end_date"), "yyyy-MM-dd")) \
    .withColumn("status", trim(col("status"))) \
    .withColumn("budget_usd", col("budget_usd").cast("double")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .dropDuplicates(["project_id"]) \
    .filter(col("project_id").isNotNull())

df_silver_project.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_PATH}.silver_project")
print(f"silver_project: {df_silver_project.count()} records")

silver_project: 400 records


In [0]:
#====================================================================================
# Silver - dim_date         
#====================================================================================

df_date = spark.table(f"{BRONZE_PATH}.bronze_date")

df_silver_date = df_date \
    .withColumn("full_date", to_date(col("full_date"), "yyyy-MM-dd")) \
    .withColumn("year", col("year").cast("integer")) \
    .withColumn("quarter", col("quarter").cast("integer")) \
    .withColumn("month", col("month").cast("integer")) \
    .withColumn("month_name", trim(col("month_name"))) \
    .withColumn("week", col("week").cast("integer")) \
    .withColumn("day_of_week", trim(col("day_of_week"))) \
    .withColumn("is_weekend", col("is_weekend").cast("boolean")) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .dropDuplicates(["date_id"]) \
    .filter(col("date_id").isNotNull())

df_silver_date.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_PATH}.silver_date")
print(f"silver_date: {df_silver_date.count()} records")

silver_date: 3653 records


In [0]:
#====================================================================================
# Silver - fact_volunteer_activity      
#====================================================================================
df_fact = spark.table(f"{BRONZE_PATH}.bronze_fact_activity")

df_silver_fact = df_fact \
    .withColumn("hours_logged", col("hours_logged").cast("double")) \
    .withColumn("beneficiaries_reached", col("beneficiaries_reached").cast("integer")) \
    .withColumn("activity_type", trim(col("activity_type"))) \
    .withColumn("outcome", trim(col("outcome"))) \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .dropDuplicates(["activity_id"]) \
    .filter(col("activity_id").isNotNull()) \
    .filter(col("volunteer_id").isNotNull()) \
    .filter(col("project_id").isNotNull()) \

df_silver_fact.write.format("delta").mode("overwrite").saveAsTable(f"{SILVER_PATH}.silver_fact_activity")

print(f"silver_fact_activity: {df_silver_fact.count()} records")

silver_fact_activity: 800000 records
